In [1]:
!pip install --upgrade pandas numpy matplotlib seaborn scipy statsmodels scikit-learn

  Using cached pandas-3.0.2-cp312-cp312-win_amd64.whl.metadata (19 kB)
Using cached pandas-3.0.2-cp312-cp312-win_amd64.whl (9.7 MB)


In [2]:
# Diagnostic Info: Check if matplotlib is healthy
try:
    import sys
    import matplotlib
    print(f"Python executable: {sys.executable}")
    print(f"Matplotlib version: {matplotlib.__version__}")
    print(f"Matplotlib file: {matplotlib.__file__}")
    print(f"Has get_data_path: {hasattr(matplotlib, 'get_data_path')}")
except Exception as e:
    print(f"Initial diagnostic failed: {e}")

# Attempt to import required libraries and provide graceful fallbacks/install hints
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from sklearn.model_selection import train_test_split
    import warnings

    warnings.filterwarnings('ignore')
    sns.set_theme(style="whitegrid", palette="muted")
    print("\nSuccessfully imported all required libraries.")

except Exception as e:
    print(f"\nWarning: Some libraries failed to load correctly: {e}")
    print("\n*** IMPORTANT: If you just ran the installation cell, please RESTART THE KERNEL now! ***")
    print("Attempting to load core libraries (pandas, numpy, matplotlib, seaborn)... ")
    try:
        import pandas as pd
        import numpy as np
        import matplotlib.pyplot as plt
        import seaborn as sns
        import scipy.stats as stats
        import warnings
        warnings.filterwarnings('ignore')
        sns.set_theme(style="whitegrid", palette="muted")
        print("Core libraries loaded. Note: Advanced statistics (statsmodels/sklearn) may be unavailable.")
    except Exception as e2:
        print(f"Critical Error: Core libraries failed to load: {e2}")
        print("Please check your Python environment or run the installation cell above.")


Python executable: c:\Users\yasan\OneDrive\Documents\sliit y3s1\IT3011_GroupAssignment\.venv\Scripts\python.exe
Matplotlib version: 3.10.8
Matplotlib file: c:\Users\yasan\OneDrive\Documents\sliit y3s1\IT3011_GroupAssignment\.venv\Lib\site-packages\matplotlib\__init__.py
Has get_data_path: True

Successfully imported all required libraries.


> This cell installs any missing Python packages required later in the notebook\. It is safe to re\-run and produces minimal output\. If packages are installed, please re\-run the import cell above to ensure fresh imports in this session\.

## Phase 1: Data Acquisition and Preprocessing
We begin by understanding our data shape, locating missing values, handling extreme outliers, and creating categorical proxy variables.

In [3]:
# 1. Load Data
data = pd.read_csv("../data/raw/data.csv")
print(f"Raw Data Shape: {data.shape}\n")

Raw Data Shape: (14003, 16)



In [4]:
# View the first 5 rows
print(data.head())

# View the column names
print(data.columns)

   StudyHours  Attendance  Resources  Extracurricular  Motivation  Internet  \
0          19          64          1                0           0         1   
1          19          64          1                0           0         1   
2          19          64          1                0           0         1   
3          19          64          1                1           0         1   
4          19          64          1                1           0         1   

   Gender  Age  LearningStyle  OnlineCourses  Discussions  \
0       0   19              2              8            1   
1       0   23              3             16            0   
2       0   28              1             19            0   
3       0   19              2              8            1   
4       0   23              3             16            0   

   AssignmentCompletion  ExamScore  EduTech  StressLevel  FinalGrade  
0                    59         40        0            1           3  
1               

In [5]:
print(data.info())

<class 'pandas.DataFrame'>
RangeIndex: 14003 entries, 0 to 14002
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   StudyHours            14003 non-null  int64
 1   Attendance            14003 non-null  int64
 2   Resources             14003 non-null  int64
 3   Extracurricular       14003 non-null  int64
 4   Motivation            14003 non-null  int64
 5   Internet              14003 non-null  int64
 6   Gender                14003 non-null  int64
 7   Age                   14003 non-null  int64
 8   LearningStyle         14003 non-null  int64
 9   OnlineCourses         14003 non-null  int64
 10  Discussions           14003 non-null  int64
 11  AssignmentCompletion  14003 non-null  int64
 12  ExamScore             14003 non-null  int64
 13  EduTech               14003 non-null  int64
 14  StressLevel           14003 non-null  int64
 15  FinalGrade            14003 non-null  int64
dtypes: int64(16)
me

In [6]:
print(data.describe())

         StudyHours    Attendance     Resources  Extracurricular  \
count  14003.000000  14003.000000  14003.000000     14003.000000   
mean      19.987431     80.194316      1.104406         0.594158   
std        5.890637     11.472181      0.697362         0.491072   
min        5.000000     60.000000      0.000000         0.000000   
25%       16.000000     70.000000      1.000000         0.000000   
50%       20.000000     80.000000      1.000000         1.000000   
75%       24.000000     90.000000      2.000000         1.000000   
max       44.000000    100.000000      2.000000         1.000000   

         Motivation      Internet        Gender           Age  LearningStyle  \
count  14003.000000  14003.000000  14003.000000  14003.000000   14003.000000   
mean       0.905806      0.925516      0.551953     23.532172       1.515461   
std        0.695896      0.262566      0.497311      3.514293       1.112941   
min        0.000000      0.000000      0.000000     18.000000      

In [7]:
columns_to_keep = ['Age', 'Gender', 'StudyHours', 'Attendance', 'AssignmentCompletion', 'ExamScore']
df = data[columns_to_keep].copy()

In [8]:
# 2. Destroy Duplicates
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_rows - len(df)} duplicate rows.")

Dropped 9803 duplicate rows.


In [9]:
# 3. Handle Missing Values
print("\nMissing values per column before cleaning:")
print(df.isnull().sum())
df = df.dropna()
print(f"Total rows remaining after dropping NaNs: {len(df)}")


Missing values per column before cleaning:
Age                     0
Gender                  0
StudyHours              0
Attendance              0
AssignmentCompletion    0
ExamScore               0
dtype: int64
Total rows remaining after dropping NaNs: 4200


In [10]:
# 4. Feature Engineering (The Buckets for Slide 5)
def categorize_completion(val):
    if val >= 80:
        return "High"
    elif val >= 50:
        return "Moderate"
    else:
        return "Low"

df['SelfAssessment_Level'] = df['AssignmentCompletion'].apply(categorize_completion)
# Lock the order so "Low" is always compared against "High" in charts
df['SelfAssessment_Level'] = pd.Categorical(df['SelfAssessment_Level'], categories=["Low", "Moderate", "High"], ordered=True)

In [11]:
# 5. Outlier Removal using Tukey's IQR on ExamScore
Q1 = df['ExamScore'].quantile(0.25)
Q3 = df['ExamScore'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df_clean = df[(df['ExamScore'] >= lower_bound) & (df['ExamScore'] <= upper_bound)]
print(f"\nDropped {len(df) - len(df_clean)} extreme outliers from ExamScore.")


Dropped 0 extreme outliers from ExamScore.


In [12]:
# 6. Save the gold-standard dataset
df_clean.to_csv("../data/processed/cleaned_data.csv", index=False)
print("Data cleaning complete. Clean file saved as 'cleaned_data.csv'.")

Data cleaning complete. Clean file saved as 'cleaned_data.csv'.


### Note: Cleaning logic consolidated above. Cells below load the cleaned dataset for analysis.

# Phase 2: Descriptive Analytics (The "Get to Know You" Phase)


In [13]:
print("Loading cleaned dataset...")
df = pd.read_csv("../data/processed/cleaned_data.csv")

# Ensure the categorical order is maintained for the charts
df['SelfAssessment_Level'] = pd.Categorical(df['SelfAssessment_Level'], categories=["Low", "Moderate", "High"], ordered=True)

Loading cleaned dataset...


In [14]:
# ---- 1. Univariate Analysis (Shape of the target variable) ----
print("\nCalculating Descriptive Statistics for Exam Score:")
print(df['ExamScore'].describe())


Calculating Descriptive Statistics for Exam Score:
count    4200.000000
mean       70.378333
std        17.714493
min        40.000000
25%        55.000000
50%        70.500000
75%        86.000000
max       100.000000
Name: ExamScore, dtype: float64


In [15]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
# Visual 1: Histogram (For Slide 6)
plt.figure(figsize=(8, 5))
sns.histplot(df['ExamScore'], kde=True, color='blue', bins=30)
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam Score")
plt.ylabel("Frequency of Students")
plt.savefig("histogram_exam_score.png") # Saves the image for your PowerPoint
plt.close()

In [17]:
# ---- 2. Bivariate Analysis (Visualizing the Hypothesis) ----
# Visual 2: Boxplot comparing Exam Scores across Self-Assessment Levels (For Slide 6)
plt.figure(figsize=(8, 5))
sns.boxplot(x='SelfAssessment_Level', y='ExamScore', data=df, palette="Set2")
plt.title("Exam Scores by Self-Assessment Level")
plt.xlabel("Self-Assessment (Assignment Completion)")
plt.ylabel("Exam Score")
plt.savefig("boxplot_exam_by_level.png") # Saves the image for your PowerPoint
plt.close()

In [18]:
# ---- 3. The Normality Test (Crucial Assumption Check) ----
print("\nChecking Normality Assumption...")
# Calculate Skewness and Kurtosis
skewness = df['ExamScore'].skew()
print(f"Skewness: {skewness:.3f} (Ideal is between -1 and 1)")

print("\nPhase 2 Complete. Visuals saved for presentation.")


Checking Normality Assumption...
Skewness: -0.037 (Ideal is between -1 and 1)

Phase 2 Complete. Visuals saved for presentation.


In [19]:
# ---- Test 1: Independent T-Test (Comparing Groups) ----
print("1. Running Independent T-Test (Moderate vs. High Self-Assessment)")

# Isolate the two groups
group_moderate = df[df['SelfAssessment_Level'] == 'Moderate']['ExamScore']
group_high = df[df['SelfAssessment_Level'] == 'High']['ExamScore']

# Run the test
t_stat, p_value_ttest = stats.ttest_ind(group_moderate, group_high, equal_var=False) # Welch's t-test handles unequal variances
print(f"T-Statistic: {t_stat:.4f}")
print(f"P-Value: {p_value_ttest:.10f}")

if p_value_ttest < 0.05:
    print("Conclusion: Reject the Null Hypothesis. There is a significant difference between groups.")
else:
    print("Conclusion: Fail to reject the Null Hypothesis. No significant difference found.")

# ---- Test 2: Pearson Correlation (Continuous Relationship) ----
print("\n2. Running Pearson Correlation (Assignment Completion vs. Exam Score)")

correlation, p_value_corr = stats.pearsonr(df['AssignmentCompletion'], df['ExamScore'])
print(f"Pearson Correlation Coefficient (r): {correlation:.4f}")
print(f"P-Value: {p_value_corr:.10f}")

if p_value_corr < 0.05:
    print("Conclusion: There is a statistically significant linear correlation.")
else:
    print("Conclusion: No significant linear correlation.")

# ---- Visual: Scatter Plot for Slide 7 ----
print("\nGenerating scatter plot for presentation...")
plt.figure(figsize=(8, 5))
sns.regplot(x='AssignmentCompletion', y='ExamScore', data=df, 
            scatter_kws={'alpha':0.3, 'color': 'blue'}, 
            line_kws={'color': 'red', 'linewidth': 2})
plt.title("Correlation: Assignment Completion vs. Exam Score")
plt.xlabel("Self-Assessment (Assignment Completion %)")
plt.ylabel("Learning Skills (Exam Score)")
plt.savefig("scatter_completion_vs_exam.png")
plt.close()
print("Saved 'scatter_completion_vs_exam.png' for Slide 7.")

1. Running Independent T-Test (Moderate vs. High Self-Assessment)
T-Statistic: -2.0788
P-Value: 0.0377025693
Conclusion: Reject the Null Hypothesis. There is a significant difference between groups.

2. Running Pearson Correlation (Assignment Completion vs. Exam Score)
Pearson Correlation Coefficient (r): 0.0195
P-Value: 0.2065215219
Conclusion: No significant linear correlation.

Generating scatter plot for presentation...
Saved 'scatter_completion_vs_exam.png' for Slide 7.


In [21]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

print("============================================================")
print("Phase 4: Multiple Linear Regression & VIF")
print("============================================================\n")


# 2. Define the Target (y) and the Predictors (X)
# We include our main proxy AND our two control variables
X = df[['AssignmentCompletion', 'StudyHours', 'Attendance']]
y = df['ExamScore']

# Add a constant (the Y-intercept) - required by statsmodels
X_with_const = sm.add_constant(X)

# ---- Step 1: The VIF Test (Checking the Multicollinearity Assumption) ----
print("--- Variance Inflation Factor (VIF) Check ---")
vif_data = pd.DataFrame()
vif_data["Variable"] = X_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]

# We drop the 'const' row for cleaner printing, as its VIF doesn't matter
print(vif_data[vif_data['Variable'] != 'const'].to_string(index=False))
print("\n(Note: A VIF less than 5 means no severe multicollinearity.)\n")

# ---- Step 2: The Multiple Linear Regression Model ----
print("--- OLS Regression Results ---")
# Build and fit the Ordinary Least Squares (OLS) model
model = sm.OLS(y, X_with_const).fit()

# Print the academic summary
print(model.summary())

Phase 4: Multiple Linear Regression & VIF

--- Variance Inflation Factor (VIF) Check ---
            Variable      VIF
AssignmentCompletion 1.000017
          StudyHours 1.000656
          Attendance 1.000645

(Note: A VIF less than 5 means no severe multicollinearity.)

--- OLS Regression Results ---
                            OLS Regression Results                            
Dep. Variable:              ExamScore   R-squared:                       0.001
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.7376
Date:                Thu, 23 Apr 2026   Prob (F-statistic):              0.529
Time:                        11:21:20   Log-Likelihood:                -18030.
No. Observations:                4200   AIC:                         3.607e+04
Df Residuals:                    4196   BIC:                         3.609e+04
Df Model:                           3                            